In [8]:
import os
import json
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import wandb
import plotly.express as px
import geopandas as gpd
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.tree import plot_tree

In [9]:
# --- PERSISTENT LOGIN LOGIC ---
if wandb.run is None:
    wandb.login()

wandb.init(
    entity="mmcgee18-georgia-state-university",
    project="Final GO-NOGO Classifier",
    name="Final-v13-Multi-Factor-Comparison",
    config={
        "test_size": 0.2,
        "n_estimators": 100,
        "random_state": 42,
        "health_weight": 0.60,
        "finance_weight": 0.40,
        "hpsa_boost": 0.15
    },
    reinit=True 
)

In [10]:
# --- DATA LOADING & SCORING (Stages 0-1) ---
# [Loading logic as before...]
hpsa_df = pd.read_csv('GitHub (New)/DS_Capstone_Group_1/data/v3_final/hpsa_need_category_v4.csv')
monte_df = pd.read_csv('GitHub (New)/DS_Capstone_Group_1/data/v3_final/monte_carlo_simulation_results.csv')
financial_df = pd.read_csv('GitHub (New)/DS_Capstone_Group_1/data/v3_final/roi_by_county_v1.csv', index_col=0)
monte_df.insert(0, 'geoid', hpsa_df['geoid'])
df = hpsa_df.merge(monte_df, on="geoid").merge(financial_df, on="geoid")

In [11]:
# --- 2. BASE DATA AGGREGATION ---
county_df = df.groupby('geoid').agg({
    'diabetes_pct_reduction_mean': 'mean', 'high_bp_pct_reduction_mean': 'mean',
    'high_cholesterol_pct_reduction_mean': 'mean', 'asthma_pct_reduction_mean': 'mean',
    'no_checkup_pct_reduction_mean': 'mean', 'roi_mean': 'mean', 'roi_std': 'mean',
    'hpsa_need_category': 'max', 'county': 'first', 'state': 'first'
}).reset_index().dropna()

county_df['fips_code'] = county_df['geoid'].astype(int).astype(str).str.zfill(5)

In [12]:
# --- 3. COMPOSITE SCORING (STAGES 1-7 RECAP) ---
scaler = MinMaxScaler()
health_cols = ['diabetes_pct_reduction_mean', 'high_bp_pct_reduction_mean', 
               'high_cholesterol_pct_reduction_mean', 'asthma_pct_reduction_mean', 
               'no_checkup_pct_reduction_mean']

scaled_health = scaler.fit_transform(county_df[health_cols])
scaled_roi = scaler.fit_transform(county_df[['roi_mean']])

# Calculating specific sub-scores for later mapping
county_df['total_health_impact'] = scaled_health.sum(axis=1) * (wandb.config.health_weight / 5)
base_finance = scaled_roi.flatten() * wandb.config.finance_weight
hpsa_multiplier = 1 + (county_df['hpsa_need_category'] * wandb.config.hpsa_boost)

county_df['composite_score'] = (county_df['total_health_impact'] + base_finance) * hpsa_multiplier
county_df['is_best'] = (county_df['composite_score'] >= county_df['composite_score'].quantile(0.75)).astype(int)

# BASE MAP PREP (Load once)
with open('us_counties.json') as f:
    counties_geojson = json.load(f)
usa_gdf = gpd.read_file('us_counties.json')
usa_gdf['id'] = usa_gdf['id'].astype(str).str.zfill(5)

In [13]:
# --- STAGE 8-9: ROI DATA ONLY ---
print("Mapping ROI Factor...")
fig_roi = px.choropleth(
    county_df, geojson=counties_geojson, locations='fips_code', color='roi_mean',
    color_continuous_scale="Reds", scope="usa", featureidkey="id", hover_name='county',
    title="Financial Performance (ROI Mean)"
)
fig_roi.update_traces(marker_line_width=0)
wandb.log({"map_roi_only": fig_roi})

# Fresh merge for static map
roi_static_gdf = usa_gdf.merge(county_df[['fips_code', 'roi_mean']], left_on='id', right_on='fips_code')
fig, ax = plt.subplots(figsize=(15, 8))
roi_static_gdf.plot(column='roi_mean', ax=ax, legend=True, cmap='Reds')
ax.set_xlim([-130, -65]); plt.axis('off'); plt.title("ROI Heatmap")
wandb.log({"static_roi_only": wandb.Image(plt)})
plt.close()

Mapping ROI Factor...


In [14]:
# --- STAGE 10-11: HEALTH METRICS ONLY ---
print("Mapping Health Factor...")
fig_health = px.choropleth(
    county_df, geojson=counties_geojson, locations='fips_code', color='total_health_impact',
    color_continuous_scale="Blues", scope="usa", featureidkey="id", hover_name='county',
    title="Health Impact Potential"
)
fig_health.update_traces(marker_line_width=0)
wandb.log({"map_health_only": fig_health})

# Fresh merge using the newly created column
health_static_gdf = usa_gdf.merge(county_df[['fips_code', 'total_health_impact']], left_on='id', right_on='fips_code')
fig, ax = plt.subplots(figsize=(15, 8))
health_static_gdf.plot(column='total_health_impact', ax=ax, legend=True, cmap='Blues')
ax.set_xlim([-130, -65]); plt.axis('off'); plt.title("Health Impact Heatmap")
wandb.log({"static_health_only": wandb.Image(plt)})
plt.close()

Mapping Health Factor...


In [15]:
# --- STAGE 12-13: HPSA NEED ONLY ---
print("Mapping HPSA Factor...")
fig_hpsa = px.choropleth(
    county_df, geojson=counties_geojson, locations='fips_code', color='hpsa_need_category',
    color_continuous_scale="Purples", scope="usa", featureidkey="id", hover_name='county',
    title="HPSA Need Category (0-3)"
)
fig_hpsa.update_traces(marker_line_width=0)
wandb.log({"map_hpsa_only": fig_hpsa})

# Fresh merge for HPSA
hpsa_static_gdf = usa_gdf.merge(county_df[['fips_code', 'hpsa_need_category']], left_on='id', right_on='fips_code')
fig, ax = plt.subplots(figsize=(15, 8))
hpsa_static_gdf.plot(column='hpsa_need_category', ax=ax, legend=True, cmap='Purples')
ax.set_xlim([-130, -65]); plt.axis('off'); plt.title("HPSA Need Heatmap")
wandb.log({"static_hpsa_only": wandb.Image(plt)})
plt.close()

Mapping HPSA Factor...


In [16]:
# --- FINAL STAGE: EXPORT ---
# county_df.to_csv('counties_for_flourish.csv', index=False)
# artifact = wandb.Artifact('full_analysis_dataset', type='dataset')
# artifact.add_file('counties_for_flourish.csv')
# wandb.log_artifact(artifact)
wandb.finish()
print("All stages complete.")

All stages complete.
